In [ ]:
# Instalasi pustaka (Colab sudah preinstall; untuk laptop local jalankan)
!pip install -q osmnx rasterio matplotlib networkx pandas

In [ ]:
import json
import time
import heapq
import random
import tracemalloc

import numpy as np
import pandas as pd
import osmnx as ox
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# =====================================================================
# KONFIGURASI GLOBAL SIMULASI
# =====================================================================
T_START_MIN  = 60.0   # agen berangkat di menit ke-60 (fase pasang naik)
V_MPS        = 8.33   # 30 km/jam -> m/s, kecepatan normal dalam kota
H_KRITIS_M   = 0.40   # > 40 cm genangan = mesin mobil mati (batas knalpot)

In [ ]:
# =====================================================================
# MEMUAT DATA SPASIAL
# =====================================================================
# 1. Graf jalan OSMnx (3 kecamatan pesisir yang terdampak rob).
# 2. Elevasi tiap simpul: spatial point-sampling join ke raster
#    DEMNAS (Ina-Geoportal, datum EGM2008).
# 3. Pasar pasang surut laut: CSV BMKG kolom EST, interval 10 menit.
#
# Saat Colab: salin chart_data.csv dan DEMNAS_1409-22_v1.0.tif
# ke folder kerja notebook (folder yang sama dengan notebook).

PATH_CSV    = "chart_data.csv"
PATH_DEMNAS = "DEMNAS_1409-22_v1.0.tif"

KECAMATAN = [
    "Semarang Utara, Semarang, Central Java, Indonesia",
    "Semarang Timur, Central Java, Indonesia",
    "Gayamsari, Semarang, Central Java, Indonesia",
]

print("Memuat graf jalan...")
G_raw = ox.graph_from_place(KECAMATAN, network_type="drive")
print(f"  nodes={len(G_raw.nodes)}, edges={len(G_raw.edges)}")

print("Spatial join elevasi DEMNAS -> simpul graf...")
n_teruji, n_fallback = 0, 0
try:
    with rasterio.open(PATH_DEMNAS) as dem:
        for node, d in G_raw.nodes(data=True):
            # rasterio.sample memakai (lon, lat) sesuai CRS geografis
            val = next(dem.sample([(d["x"], d["y"])]))[0]
            if np.isnan(val) or not (0.0 <= val <= 150.0):
                G_raw.nodes[node]["elev"] = 1.0   # piksel NoData: pesisir datar
                n_fallback += 1
            else:
                G_raw.nodes[node]["elev"] = float(val)
                n_teruji += 1
    print(f"  elevasi valid: {n_teruji}, fallback: {n_fallback}")
except FileNotFoundError:
    print("  [!] DEMNAS tidak ditemukan; semua simpul pakai elev dummy 1.2 m")
    for node in G_raw.nodes:
        G_raw.nodes[node]["elev"] = 1.2

# Graf proyeksi UTM (meter) untuk perhitungan jarak fisik
G = ox.project_graph(G_raw)

print("Memuat data pasang surut (CSV BMKG, kolom EST)...")
try:
    df = pd.read_csv(PATH_CSV)
    # 1 baris = interval 10 menit; mapping menit -> tinggi air laut (m)
    df["menit"] = np.arange(len(df)) * 10
    EST = dict(zip(df["menit"], df["EST"]))
    print(f"  {len(EST)} interval 10-menit dimuat")
except FileNotFoundError:
    print("  [!] CSV pasang surut tidak ditemukan; pakai flat 1.55 m")
    EST = {m: 1.55 for m in range(0, 1440, 10)}

In [ ]:
# =====================================================================
# MODEL FISIS BANJIR ROB + TIGA ALGORITMA PENCARIAN RUTE
# =====================================================================

def rob_depth(node, menit, graf):
    """Kedalaman genangan (m) di simpul pada `menit`.

    Kedalaman = tinggi air laut EST - elevasi tanah simpul,
    dibatasi minimum 0 (tanah lebih tinggi daripada air laut).
    """
    elev = graf.nodes[node]["elev"]
    menit_10 = int(round(menit / 10.0) * 10)     # grid interval 10 menit
    menit_10 = min(menit_10, max(EST))
    return max(0.0, EST.get(menit_10, 0.0) - elev)


def w_penalty(d):
    """Faktor penalti w(d): pengali jarak efektif antar simpul.

       d < 0.20 m        -> 1.0   (jalan kering, normal)
       0.20 <= d <= 0.40 -> 2.5   (mobil melambat, genangan dangkal)
       d > 0.40 m        -> inf   (mesin mati, jalan buntu -> pangkas)
    """
    if d < 0.20:
        return 1.0
    if d <= H_KRITIS_M:
        return 2.5
    return float("inf")


def h_euclid(graf, a, b):
    """Heuristik admissible: jarak Euclidean planar UTM (m) ke simpul tujuan."""
    x1, y1 = graf.nodes[a]["x"], graf.nodes[a]["y"]
    x2, y2 = graf.nodes[b]["x"], graf.nodes[b]["y"]
    return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5


def flood_exposure(path, t0, graf):
    """Total kedalaman genangan (m) yang 'disentuh' mobil sepanjang rute.

    Semakin besar skor = semakin berat paparan banjir kendaraan.
    """
    if not path:
        return 0.0
    total, t = 0.0, t0
    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        L = graf[u][v][0]["length"]
        d = rob_depth(v, t, graf)
        total += d
        w = w_penalty(d)
        v_eff = V_MPS / (w if w != float("inf") else 100.0)
        t += L / v_eff
    return total


# ---------- A* STANDAR (baseline, buta banjir) ----------
# Bobot busur = panjang geometris statis; tidak peka waktu/banjir.
def astar_standard(graf, s, goal, t0):
    open_ = [(h_euclid(graf, s, goal), s, t0)]
    g = {n: float("inf") for n in graf}
    g[s] = 0.0
    explored, parent = set(), {}
    while open_:
        _, u, t = heapq.heappop(open_)
        explored.add(u)
        if u == goal:
            path = [goal]
            while path[-1] != s:
                path.append(parent[path[-1]])
            return path[::-1], explored
        for v in graf[u]:
            L = graf[u][v][0]["length"]
            tg = g[u] + L
            if tg < g.get(v, float("inf")):
                g[v] = tg
                parent[v] = u
                heapq.heappush(open_, (tg + h_euclid(graf, v, goal), v, t + L / V_MPS))
    return None, explored


# ---------- BRANCH & BOUND (kontrol eksak) ----------
# DFS + pruning: buang cabang bila biaya sudah melewati best,
# atau bila simpul tetangga tenggelam > H_KRITIS_M.
def branch_and_bound(graf, s, goal, t0):
    best_cost, best_path = float("inf"), None
    seen = {n: float("inf") for n in graf}
    seen[s] = 0.0
    explored = 0
    stack = [(0.0, s, t0, [s])]
    while stack:
        c, u, t, path = stack.pop()
        explored += 1
        if u == goal:
            if c < best_cost:
                best_cost, best_path = c, path
            continue
        for v in graf[u]:
            L = graf[u][v][0]["length"]
            w = w_penalty(rob_depth(v, t, graf))
            if w == float("inf"):
                continue
            nc = c + L * w
            if nc >= best_cost or nc >= seen.get(v, float("inf")):
                continue
            seen[v] = nc
            stack.append((nc, v, t + L / (V_MPS / w), path + [v]))
    return best_path, explored


# ---------- CUSTOM TIME-DEPENDENT A* (USULAN) ----------
# Seperti A* standar, tetapi g-score dikali w(d) dan waktu ketibaan
# t dipakai hitung d tiap ekspansi. Saat air naik, w melonjak ->
# cabang tenggelam tidak lagi seksi.
def astar_time_dependent(graf, s, goal, t0):
    open_ = [(h_euclid(graf, s, goal), s, t0)]
    g = {n: float("inf") for n in graf}
    g[s] = 0.0
    explored, parent = set(), {}
    while open_:
        _, u, t = heapq.heappop(open_)
        explored.add(u)
        if u == goal:
            path = [goal]
            while path[-1] != s:
                path.append(parent[path[-1]])
            path = path[::-1]
            # rekonstruksi timestamp per simpul untuk visualisasi
            ts, tt = [t0], t0
            for i in range(len(path) - 1):
                L = graf[path[i]][path[i + 1]][0]["length"]
                w = w_penalty(rob_depth(path[i + 1], tt, graf))
                tt += L / (V_MPS / w)
                ts.append(tt)
            return path, explored, ts
        for v in graf[u]:
            L = graf[u][v][0]["length"]
            w = w_penalty(rob_depth(v, t, graf))
            if w == float("inf"):
                continue
            tg = g[u] + L * w
            if tg < g.get(v, float("inf")):
                g[v] = tg
                parent[v] = u
                heapq.heappush(open_, (tg + h_euclid(graf, v, goal), v, t + L / (V_MPS / w)))
    return None, explored, []


# Posko evakuasi resmi (koordinat lon, lat) -> simpul graf terdekat
SHELTERS = {
    "P_Rejomulyo_Balai": (110.4349, -6.9643),
    "P8_Kemijen":        (110.4387, -6.9594),
}
shelter_nodes = {n: ox.nearest_nodes(G_raw, X=lon, Y=lat)
                 for n, (lon, lat) in SHELTERS.items()}

In [ ]:
# =====================================================================
# SAMPLING AGEN: 1 MOBIL DARI ZONA BERESIKO TINGGI SEMARANG UTARA
# =====================================================================
# Titik asal = simpul acak di bounding box pesisir Semarang Utara
# dengan elevasi <= 1.0 m (rawan tenggelam paling dulu saat rob).
# Seed dikunci agar hasil reproducible.

bbox = dict(lat_lo=-6.9600, lat_hi=-6.9350, lon_lo=110.4000, lon_hi=110.4400)
candidates = [
    n for n, d in G_raw.nodes(data=True)
    if bbox["lat_lo"] <= d["y"] <= bbox["lat_hi"]
    and bbox["lon_lo"] <= d["x"] <= bbox["lon_hi"]
    and d.get("elev", 99) <= 1.0
]
if not candidates:
    print("[!] bounding box kosong; pakai semua simpul elev <= 1.0 m")
    candidates = [n for n, d in G_raw.nodes(data=True) if d.get("elev", 99) <= 1.0]

random.seed(12)
START = random.choice(candidates)
GOAL  = shelter_nodes["P_Rejomulyo_Balai"]

# Simpan sebagai dataset agen (bisa ditambah agen lain)
with open("dataset_mini_evakuasi.json", "w") as f:
    json.dump([{"id_agen": 1, "node_start": int(START), "node_destination": int(GOAL)}],
              f, indent=2)

print(f"START = {START}  (elev {G_raw.nodes[START]['elev']:.2f} m)")
print(f"GOAL  = {GOAL}   (elev {G_raw.nodes[GOAL]['elev']:.2f} m)")

In [ ]:
# =====================================================================
# EKSEKUSI 3 ALGORITMA + REKAP METRIK
# =====================================================================
agents = json.load(open("dataset_mini_evakuasi.json"))

def run_timed(fn, *args):
    """Jalankan `fn` sekali, kembalikan (hasil, wall-time ms, peak memory KB)."""
    tracemalloc.start()
    t0 = time.perf_counter()
    out = fn(*args)
    dt_ms = (time.perf_counter() - t0) * 1000
    peak_kb = tracemalloc.get_traced_memory()[1] / 1024
    tracemalloc.stop()
    return out, dt_ms, peak_kb

def path_km(graf, p):
    return sum(graf[p[i]][p[i + 1]][0]["length"] for i in range(len(p) - 1)) / 1000.0

rows = []          # untuk tabel metrik
fig_data = {}      # untuk plotting sel berikutnya

for a in agents:
    s, g = a["node_start"], a["node_destination"]

    # --- 1. A* standar (buta banjir) ---
    (p_sa, exp_sa), dt_sa, mem_sa = run_timed(astar_standard, G, s, g, T_START_MIN)
    ok_sa = max(rob_depth(n, T_START_MIN, G) for n in p_sa) <= H_KRITIS_M if p_sa else False
    rows.append(["Standard_A*", dt_sa, len(exp_sa), mem_sa,
                 path_km(G, p_sa), flood_exposure(p_sa, T_START_MIN, G), ok_sa])
    fig_data["sa"] = {"path": p_sa, "explored": exp_sa}

    # --- 2. Custom TD-A* (usulan) ---
    (p_td, exp_td, ts_td), dt_td, mem_td = run_timed(
        astar_time_dependent, G, s, g, T_START_MIN)
    ok_td = max(rob_depth(n, T_START_MIN, G) for n in p_td) <= H_KRITIS_M if p_td else False
    rows.append(["Custom_TD_A*", dt_td, len(exp_td), mem_td,
                 path_km(G, p_td), flood_exposure(p_td, T_START_MIN, G), ok_td])
    fig_data["td"] = {"path": p_td, "explored": exp_td, "ts": ts_td}

    # --- 3. Branch & Bound (kontrol eksak) ---
    # B&B eksak di graf 4400 simpul memang lambat; dipakai sebagai
    # reference saja (lihat pembahasan di makalah).
    tracemalloc.start()
    t0_bb = time.perf_counter()
    p_bb, exp_bb = branch_and_bound(G, s, g, T_START_MIN)
    dt_bb = (time.perf_counter() - t0_bb) * 1000
    mem_bb = tracemalloc.get_traced_memory()[1] / 1024
    tracemalloc.stop()
    ok_bb = p_bb is not None
    rows.append(["Branch_Bound", dt_bb, exp_bb, mem_bb,
                 path_km(G, p_bb) if p_bb else 0.0,
                 flood_exposure(p_bb, T_START_MIN, G) if p_bb else 0.0, ok_bb])
    fig_data["bb"] = {"path": p_bb, "explored_count": exp_bb}

df = pd.DataFrame(
    rows,
    columns=["Algo", "Time_ms", "Explored_nodes", "Peak_mem_KB",
             "Path_km", "Flood_exposure_m", "Success"])
display(df)

# Data deskriptif geografis agen
print("=" * 60)
print(f"START node {START}:  lat {G_raw.nodes[START]['y']:.6f}, "
      f"lon {G_raw.nodes[START]['x']:.6f}, elev {G_raw.nodes[START]['elev']:.2f} m")
print(f"GOAL  node {GOAL}:  lat {G_raw.nodes[GOAL]['y']:.6f}, "
      f"lon {G_raw.nodes[GOAL]['x']:.6f}, elev {G_raw.nodes[GOAL]['elev']:.2f} m")

In [ ]:
# =====================================================================
# FIGURE 3: VISUALISASI RUANG DATA (GEOPATIAL CORE)
# =====================================================================
# 2x2 panel: (a) graf jalan, (b) raster DEMNAS, (c) heatmap elevasi,
# (d) heatmap genangan rob pada saat EST puncak.
import rasterio.plot

# waktu puncak pasang surut (EST maksimum)
t_puncak = max(EST, key=EST.get)
d_puncak = [rob_depth(n, t_puncak, G) for n in G.nodes]

node_x = [G.nodes[n]["x"] for n in G.nodes]
node_y = [G.nodes[n]["y"] for n in G.nodes]
node_elev = [G.nodes[n]["elev"] for n in G.nodes]

fig, ax = plt.subplots(2, 2, figsize=(18, 16))
a1, a2, a3, a4 = ax.flatten()

# (a) graf jalan
ox.plot_graph(G, ax=a1, node_size=1, edge_color="#e0e0e0",
              node_color="#1f77b4", bgcolor="white")
a1.set_title("(a) Graf jalan berarah (OSMnx drive)")
a1.axis("off")

# (b) raster DEMNAS
try:
    with rasterio.open(PATH_DEMNAS) as dem:
        rasterio.plot.show(dem, ax=a2, cmap="terrain")
        a2.set_xlim(G_raw.nodes[0]["x"] - 0.005, G_raw.nodes[0]["x"] + 0.005)
        a2.set_title("(b) Raster DEMNAS (topografi)")
except Exception:
    a2.text(0.5, 0.5, "DEMNAS tidak tersedia\n(fallback mode)",
            ha="center", va="center", color="red", transform=a2.transAxes)
    a2.set_title("(b) Raster DEMNAS (topografi)")

# (c) heatmap elevasi
ox.plot_graph(G, ax=a3, node_size=0, edge_color="#eeeeee")
sc = a3.scatter(node_x, node_y, c=node_elev, cmap="terrain", s=12)
fig.colorbar(sc, ax=a3, label="Elevasi tanah (m EGM2008)")
a3.set_title("(c) Overlay graf + heatmap elevasi")
a3.axis("off")

# (d) heatmap genangan rob saat EST puncak
ox.plot_graph(G, ax=a4, node_size=0, edge_color="#e0e0e0")
sc = a4.scatter(node_x, node_y, c=d_puncak, cmap="Blues", s=15, vmin=0.01, vmax=1.5)
fig.colorbar(sc, ax=a4, label="Kedalaman genangan (m)")
a4.set_title(f"(d) Genangan rob saat EST puncak "
             f"(menit {t_puncak}, {EST[t_puncak]:.2f} m)")
a4.axis("off")

plt.tight_layout()
fig.savefig("Figure_3_komponen_spasial.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================================
# FIGURE 5 & 6: KOMPARASI ROUTE + ANALISIS RUNTIME LINIAR
# =====================================================================
node_x_map = {n: G.nodes[n]["x"] for n in G.nodes}
node_y_map = {n: G.nodes[n]["y"] for n in G.nodes}

# ---------- FIGURE 5: 3x2 panel (route + ruang jelajah per algo) ----------
algo_panels = [
    ("Standard_A*",   "sa", "#d62728", "Jalur buta banjir"),
    ("Branch_Bound",   "bb", "#7f7f7f", "Jalur eksak memutar"),
    ("Custom_TD_A*",   "td", "#1f77b4", "Jalur adaptif TD-A*"),
]
fig5, axes = plt.subplots(3, 2, figsize=(16, 21))
for i, (name, key, color, label) in enumerate(algo_panels):
    ar, ae = axes[i]
    path = fig_data[key]["path"]

    ox.plot_graph(G, ax=ar, node_size=0, edge_color="#e6e6e6")
    if path:
        xs = [node_x_map[n] for n in path]
        ys = [node_y_map[n] for n in path]
        ar.plot(xs, ys, color=color, lw=3, label=label, zorder=5)
        ar.scatter([xs[0]], [ys[0]], c="orange", marker="o", s=120, zorder=6)
        ar.scatter([xs[-1]], [ys[-1]], c="green", marker="*", s=220, zorder=6)
    ar.legend(fontsize=9)
    ar.set_title(f"({chr(97 + i*2)}) {name} - rute", loc="left")
    ar.axis("off")

    ox.plot_graph(G, ax=ae, node_size=0, edge_color="#eeeeee")
    ex = fig_data[key].get("explored", [])
    if ex:
        xs = [node_x_map[n] for n in ex]
        ys = [node_y_map[n] for n in ex]
        ae.scatter(xs, ys, c="#ff7f0e", s=6, alpha=0.5)
    ae.set_title(f"({chr(98 + i*2)}) {name} - ruang jelajah", loc="left")
    ae.axis("off")

plt.tight_layout()
fig5.savefig("Figure_5_komparasi_spasial.png", dpi=300, bbox_inches="tight")
plt.show()

# ---------- FIGURE 6: lintasan depth + faktor penalti w vs waktu ----------
fig6, (g1, g2) = plt.subplots(1, 2, figsize=(16, 5))

for name, key, color, style in [
    ("Standard_A*", "sa", "#d62728", "--"),
    ("Custom_TD_A*", "td", "#1f77b4", "-"),
]:
    path = fig_data[key]["path"]
    ts = fig_data[key].get("ts", [])
    if not path or not ts:
        continue
    d_series = [rob_depth(n, t, G) for n, t in zip(path, ts)]
    w_series = [5.0 if w == float("inf") else w
                for w in (w_penalty(d) for d in d_series)]
    g1.plot([t - T_START_MIN for t in ts], d_series,
            color=color, ls=style, lw=2, label=name)
    g2.step([t - T_START_MIN for t in ts], w_series,
            color=color, ls=style, lw=2, label=name)

g1.axhline(H_KRITIS_M, color="purple", ls=":", lw=1.5,
           label=f"batas knalpot {H_KRITIS_M} m")
g1.set_xlabel("Menit sejak keberangkatan")
g1.set_ylabel("Kedalaman genangan (m)")
g1.set_title("Kedalaman air vs waktu perjalanan")
g1.legend()
g1.grid(ls=":", alpha=0.5)

g2.set_xticks([0, 30, 60, 90, 120])
g2.set_yticks([1.0, 2.5, 5.0])
g2.set_yticklabels(["1.0 (kering)", "2.5 (melambat)", "inf (buntu)"])
g2.set_xlabel("Menit sejak keberangkatan")
g2.set_ylabel("Faktor penalti w")
g2.set_title("Faktor penalti w vs waktu")
g2.legend()
g2.grid(ls=":", alpha=0.5)

plt.tight_layout()
fig6.savefig("Figure_6_analisis_trajektori.png", dpi=300, bbox_inches="tight")
plt.show()
print("Selesai. Cek file PNG di folder kerja.")